# 2.5 Data quality

Data summary report, auto-generated tests from reference, and custom tests with Evidently 0.7.

In [ ]:
import warnings

import numpy as np
import pandas as pd
from evidently import DataDefinition, Dataset, Report

# Single metrics – each one can carry its own test conditions
from evidently.metrics import (
    DuplicatedRowCount,
    MeanValue,
    MinValue,
    MissingValueCount,
    OutListValueCount,
    OutRangeValueCount,
    RowCount,
)
from evidently.presets import DataSummaryPreset

# Test conditions: eq, gte, lte, ... and Reference() = "close to the reference value"
from evidently.tests import Reference, eq, gte, lte
from sklearn import datasets

warnings.filterwarnings("ignore")

## Data

In [ ]:
# Adult (census income): numerical + categorical columns, target = income >50K
adult = datasets.fetch_openml(name="adult", version=2, as_frame=True).frame

numerical = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
categorical = ["workclass", "education", "marital-status", "occupation",
               "relationship", "race", "sex", "native-country", "class"]

# Reference and current = different education groups, so there is a natural shift
groups = ["Some-college", "HS-grad", "Bachelors"]
reference = adult[~adult.education.isin(groups)]
current = adult[adult.education.isin(groups)].sample(3000, random_state=0).reset_index(drop=True)

Break the current batch on purpose, like a buggy pipeline would.

In [ ]:
current.loc[:299, "age"] = np.nan                                 # 10% missing ages (lost logging)
current.loc[300:309, "age"] = -1                                  # impossible values (bad default)
current = pd.concat([current, current.head(50)], ignore_index=True)  # duplicated rows (double load)
current["hours-per-week"] = current["hours-per-week"] * 1.5       # wrong unit / formula change

In [ ]:
# Tell Evidently the type of each column
definition = DataDefinition(numerical_columns=numerical, categorical_columns=categorical)

ref_ds = Dataset.from_pandas(reference, data_definition=definition)
cur_ds = Dataset.from_pandas(current, data_definition=definition)

## Data summary report

`DataSummaryPreset` = row/column counts, missing values, duplicates, constant columns, and per-column stats.

**`include_tests=True`** – the report not only *measures* each value, it also gives it a **PASS / FAIL**.

| | Without | With |
|---|---|---|
| Output | `Missing ages: 300` | `Missing ages: 300  FAIL` |

What counts as PASS:

- **With reference** – value must be close to the reference (about ±10%).
- **Without reference** – common-sense rules: no missing values, no duplicates, no empty or constant columns.

Why: in production nobody reads every report – code checks the FAILs and sends an alert.

In [ ]:
summary = Report([DataSummaryPreset()], include_tests=True)
summary_result = summary.run(current_data=cur_ds, reference_data=ref_ds)
summary_result

In [ ]:
# Test results as a table: what failed?
tests = pd.DataFrame(summary_result.dict()["tests"])[["name", "status"]]
tests["status"] = tests["status"].astype(str).str.replace("TestStatus.", "")
print(tests.status.value_counts())
tests[tests.status == "FAIL"]

Auto tests are strict. Row count fails only because batches have a different size than the reference,
and many stats fail because the education groups differ. Useful as a first pass, but tune them.

## Custom tests

Pick the checks that matter and set the conditions yourself.

In [ ]:
known_workclass = reference.workclass.dropna().unique().tolist()

custom = Report([
    RowCount(tests=[gte(1000)]),                                   # batch not too small
    DuplicatedRowCount(tests=[eq(0)]),                             # no duplicates
    MissingValueCount(column="age", share_tests=[lte(0.05)]),      # ≤ 5% missing
    MinValue(column="age", tests=[gte(17)]),                       # no impossible ages
    OutRangeValueCount(column="age", left=17, right=90, tests=[eq(0)]),
    MeanValue(column="hours-per-week", tests=[eq(Reference(relative=0.1))]),  # within ±10% of reference
    OutListValueCount(column="workclass", values=known_workclass, tests=[eq(0)]),  # no new categories
])
custom_result = custom.run(current_data=cur_ds, reference_data=ref_ds)
custom_result

In [ ]:
# Short summary: one line per test
for t in custom_result.dict()["tests"]:
    print(str(t["status"]).replace("TestStatus.", "").ljust(8), t["description"])

## Output

In [ ]:
summary_result.save_html("data_quality_report.html")
custom_result.save_html("data_quality_tests.html")